In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ai-tudy/pipeline_detected_family_image

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ai-tudy/pipeline_detected_family_image


In [ ]:
!pip install facenet-pytorch

In [ ]:
import sys
sys.path.insert(0, './utils/')
from downloading_from_coco_2017 import download_image_from_coco_2017_for_model_has_human
from dataset import ConvDataset
from trainer import train_model, get_y_true_pred
from other_utils import get_model_resnet18, view_classification_report, load_model, save_json
from models import (download_models_MTCNN, CustomFaceDetectionModel, evaluate_pipeline_on_samples,
                    FamilyImageFilterPipeline, create_FamilyImageFilterPipeline)
from PIL import Image
from pathlib import Path
import pandas as pd
import json
import torch
import torch.nn as nn
import random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import os
from tqdm import tqdm
from copy import deepcopy
from sklearn.metrics import confusion_matrix, classification_report



In [ ]:
download_models_MTCNN()

Скачивание предобученной архитектуры MTCNN...
Веса успешно сохранены в локальные файлы .pt в папку models/face_detector/!


In [ ]:
# 1. Загружаем ваш кастомный датасет
test_dataset = ConvDataset(root_dir="data/test/pipeline")

In [ ]:
# 2. Инициализируем пайплайн
model_human_path = 'models/has_human/best_model_has_human.pth'
model_blur_path = 'models/blur_models/best_model_blur_Adam.pth'
pipeline = create_FamilyImageFilterPipeline(model_human_path, model_blur_path)

Пайплайн инициализирован. Результаты этого прогона будут сохранены в: result/result_0


In [ ]:
# 3. Считаем предсказания
y_true, y_pred = evaluate_pipeline_on_samples(pipeline, test_dataset.samples)

Анализ тестового датасета напрямую через samples (167 изображений)...


100%|██████████| 167/167 [02:21<00:00,  1.18it/s]


In [ ]:
# 4. Выводим результаты, которые вы скопируете на слайды
print("\n" + "="*60)
print("             МЕТРИКИ КАЧЕСТВА")
print("="*60)
print(classification_report(y_true, y_pred, target_names=['Defect (0)', 'Correct (1)']))
print("Матрица ошибок (Confusion Matrix):")
print(confusion_matrix(y_true, y_pred))
print("="*60)


             МЕТРИКИ КАЧЕСТВА
              precision    recall  f1-score   support

  Defect (0)       0.66      0.92      0.77        87
 Correct (1)       0.84      0.47      0.61        80

    accuracy                           0.71       167
   macro avg       0.75      0.70      0.69       167
weighted avg       0.75      0.71      0.69       167

Матрица ошибок (Confusion Matrix):
[[80  7]
 [42 38]]
